In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp, lit, current_date, sha2, concat_ws, col, trim, to_json, struct
import uuid

# =========================================================
# CONFIG
# =========================================================
pipeline_name = "retail_pipeline"
layer_name = "bronze"
load_type = "FULL"
triggered_by = "manual"
source_system = "S3-Retail-Lakehouse-Bucket"
base_path = "/Volumes/retail/default/raw_data"

audit_table = "retail.audit.etl_run_log"
reject_table = "retail.audit.etl_reject_log"

batch_id = "batch_full_20260422"

table_configs = [
    {
        "table_name": "ns_brands",
        "file_name": "ns_brands_2026-04-22.parquet",
        "target_table": "retail.bronze.ns_brands",
        "business_key": "internal_id",
        "cast_cols": [
            "internal_id", "brand_code", "brand_name", "is_active", "created_at"
        ],
        "hash_cols": [
            "internal_id", "brand_code", "brand_name", "is_active", "created_at"
        ],
        "required_cols": ["internal_id", "brand_code", "brand_name", "is_active", "created_at"]
    },
    {
        "table_name": "ns_customers",
        "file_name": "ns_customers_2026-04-22.parquet",
        "target_table": "retail.bronze.ns_customers",
        "business_key": "internal_id",
        "cast_cols": [
            "internal_id", "entity_id", "company_name", "customer_type", "email", "phone",
            "subsidiary", "currency_code", "terms_name", "vat_number",
            "billing_address1", "billing_city", "billing_state", "billing_country",
            "shipping_address1", "shipping_city", "shipping_state", "shipping_country",
            "is_active", "created_at"
        ],
        "hash_cols": [
            "internal_id", "entity_id", "company_name", "customer_type", "email", "phone",
            "subsidiary", "currency_code", "terms_name", "vat_number",
            "billing_address1", "billing_city", "billing_state", "billing_country",
            "shipping_address1", "shipping_city", "shipping_state", "shipping_country",
            "is_active", "created_at"
        ],
        "required_cols": ["internal_id", "entity_id", "company_name", "customer_type", "subsidiary", "currency_code", "terms_name", "is_active", "created_at"]
    },
    {
        "table_name": "ns_items",
        "file_name": "ns_items_2026-04-22.parquet",
        "target_table": "retail.bronze.ns_items",
        "business_key": "internal_id",
        "cast_cols": [
            "internal_id", "item_id", "item_name", "item_type", "category",
            "subcategory", "brand_internal_id", "base_price", "cost_price",
            "is_active", "created_at"
        ],
        "hash_cols": [
            "internal_id", "item_id", "item_name", "item_type", "category",
            "subcategory", "brand_internal_id", "base_price", "cost_price",
            "is_active", "created_at"
        ],
        "required_cols": ["internal_id", "item_id", "item_name", "item_type", "category", "subcategory", "base_price", "cost_price", "is_active", "created_at"]
    },
    {
        "table_name": "ns_sales_orders",
        "file_name": "ns_sales_orders_2026-04-22.parquet",
        "target_table": "retail.bronze.ns_sales_orders",
        "business_key": "internal_id",
        "cast_cols": [
            "internal_id", "tran_id", "customer_internal_id", "order_date", "order_status",
            "sales_channel", "payment_status", "currency_code", "external_ref_num",
            "integration_source", "order_total", "created_at"
        ],
        "hash_cols": [
            "internal_id", "tran_id", "customer_internal_id", "order_date", "order_status",
            "sales_channel", "payment_status", "currency_code", "external_ref_num",
            "integration_source", "order_total", "created_at"
        ],
        "required_cols": ["internal_id", "tran_id", "customer_internal_id", "order_date", "order_status", "sales_channel", "payment_status", "currency_code", "integration_source", "order_total", "created_at"]
    },
    {
        "table_name": "ns_sales_order_lines",
        "file_name": "ns_sales_order_lines_2026-04-22.parquet",
        "target_table": "retail.bronze.ns_sales_order_lines",
        "business_key": "internal_id",
        "cast_cols": [
            "internal_id", "sales_order_internal_id", "line_num", "item_internal_id",
            "quantity", "unit_rate", "line_amount", "created_at"
        ],
        "hash_cols": [
            "internal_id", "sales_order_internal_id", "line_num", "item_internal_id",
            "quantity", "unit_rate", "line_amount", "created_at"
        ],
        "required_cols": ["internal_id", "sales_order_internal_id", "line_num", "item_internal_id", "quantity", "unit_rate", "line_amount", "created_at"]
    },
    {
        "table_name": "ns_shipments",
        "file_name": "ns_shipments_2026-04-22.parquet",
        "target_table": "retail.bronze.ns_shipments",
        "business_key": "internal_id",
        "cast_cols": [
            "internal_id", "shipment_number", "sales_order_internal_id", "warehouse_name",
            "shipment_date", "delivery_date", "shipment_status", "tracking_number", "created_at"
        ],
        "hash_cols": [
            "internal_id", "shipment_number", "sales_order_internal_id", "warehouse_name",
            "shipment_date", "delivery_date", "shipment_status", "tracking_number", "created_at"
        ],
        "required_cols": ["internal_id", "shipment_number", "sales_order_internal_id", "warehouse_name", "shipment_date", "shipment_status", "created_at"]
    }
]

# =========================================================
# AUDIT HELPERS
# =========================================================
def log_ingested_file(
    run_id,
    table_name,
    file_path,
    rows_read,
    rows_written
):
    spark.sql(f"""
        INSERT INTO retail.audit.ingested_files
        SELECT
          uuid() AS file_id,
          '{run_id}' AS run_id,
          '{batch_id}' AS batch_id,
          '{pipeline_name}' AS pipeline_name,
          '{layer_name}' AS layer_name,
          '{table_name}' AS table_name,
          '{source_system}' AS source_system,
          '{file_path}' AS source_file_path,
          split('{file_path}', '/')[size(split('{file_path}', '/'))-1] AS source_file_name,
          'SUCCESS' AS ingestion_status,
          CAST({rows_read} AS BIGINT) AS rows_read,
          CAST({rows_written} AS BIGINT) AS rows_written,
          current_timestamp() AS ingested_at
    """)

def generate_run_id():
    return str(uuid.uuid4())

def start_audit(run_id, table_name):
    spark.sql(f"""
        INSERT INTO {audit_table}
        (
          run_id, batch_id, pipeline_name, layer_name, table_name, source_system,
          load_type, triggered_by, start_time, end_time, duration_seconds,
          status, rows_read, rows_written, rows_rejected, records_before,
          records_after, error_message, created_at
        )
        VALUES (
          '{run_id}', '{batch_id}', '{pipeline_name}', '{layer_name}', '{table_name}', '{source_system}',
          '{load_type}', '{triggered_by}', current_timestamp(), NULL, NULL,
          'STARTED', NULL, NULL, NULL, NULL,
          NULL, NULL, current_timestamp()
        )
    """)

def end_audit_success(run_id, rows_read, rows_written, rows_rejected, records_before, records_after):
    spark.sql(f"""
        UPDATE {audit_table}
        SET
          end_time = current_timestamp(),
          duration_seconds = unix_timestamp(current_timestamp()) - unix_timestamp(start_time),
          status = 'SUCCESS',
          rows_read = {rows_read},
          rows_written = {rows_written},
          rows_rejected = {rows_rejected},
          records_before = {records_before},
          records_after = {records_after}
        WHERE run_id = '{run_id}'
    """)

def end_audit_fail(run_id, error_message):
    safe_error = error_message.replace("'", "''")[:5000]
    spark.sql(f"""
        UPDATE {audit_table}
        SET
          end_time = current_timestamp(),
          duration_seconds = unix_timestamp(current_timestamp()) - unix_timestamp(start_time),
          status = 'FAILED',
          error_message = '{safe_error}'
        WHERE run_id = '{run_id}'
    """)

def log_rejects(df_rejects, run_id, table_name):
    if df_rejects.isEmpty():
        return

    reject_out = (
        df_rejects
        .withColumn("reject_id", F.expr("uuid()"))
        .withColumn("run_id", lit(run_id))
        .withColumn("batch_id", lit(batch_id))
        .withColumn("pipeline_name", lit(pipeline_name))
        .withColumn("layer_name", lit(layer_name))
        .withColumn("table_name", lit(table_name))
        .withColumn("source_system", lit(source_system))
        .withColumn("rejected_at", current_timestamp())
        .withColumn("created_at", current_timestamp())
        .select(
            "reject_id",
            "run_id",
            "batch_id",
            "pipeline_name",
            "layer_name",
            "table_name",
            "source_system",
            "source_record_id",
            "reject_stage",
            "reject_reason",
            "reject_column",
            "severity",
            "raw_payload",
            "error_code",
            "error_message",
            "rejected_at",
            "created_at"
        )
    )

    reject_out.write.format("delta").mode("append").saveAsTable(reject_table)

# =========================================================
# LOAD FUNCTION
# =========================================================
def load_to_bronze(cfg):
    table_name = cfg["table_name"]
    file_name = cfg["file_name"]
    target_table = cfg["target_table"]
    cast_cols = cfg["cast_cols"]
    hash_cols = cfg["hash_cols"]
    required_cols = cfg["required_cols"]
    business_key = cfg["business_key"]

    run_id = generate_run_id()
    start_audit(run_id, table_name)

    try:
        file_path = f"{base_path}/{file_name}"

        records_before = spark.table(target_table).count()

        df = spark.read.parquet(file_path)
        rows_read = df.count()

        # cast raw columns to string
        for c in cast_cols:
            df = df.withColumn(c, col(c).cast("string"))

        # basic reject logic: required columns null/blank
        reject_condition = None
        for c in required_cols:
            cond = col(c).isNull() | (trim(col(c)) == "")
            reject_condition = cond if reject_condition is None else (reject_condition | cond)

        df_rejects = (
            df.filter(reject_condition)
            .withColumn("source_record_id", col(business_key).cast("string"))
            .withColumn("reject_stage", lit("BRONZE"))
            .withColumn("reject_reason", lit("MISSING_REQUIRED_VALUE"))
            .withColumn("reject_column", lit(",".join(required_cols)))
            .withColumn("severity", lit("ERROR"))
            .withColumn("raw_payload", to_json(struct(*[col(c) for c in df.columns])))
            .withColumn("error_code", lit("BRZ_001"))
            .withColumn("error_message", lit("One or more required columns are null or blank"))
        )

        df_valid = df.filter(~reject_condition)

        rows_rejected = df_rejects.count()

        log_rejects(df_rejects, run_id, table_name)

        # add Bronze metadata
        df_final = (
            df_valid
            .withColumn("etl_loaded_at", current_timestamp().cast("string"))
            .withColumn("source_system", lit(source_system))
            .withColumn("etl_run_id", lit(run_id))
            .withColumn("ingestion_date", current_date())
            .withColumn(
                "record_hash",
                sha2(
                    concat_ws(
                        "||",
                        *[F.coalesce(col(c).cast("string"), lit("")) for c in hash_cols]
                    ),
                    256
                )
            )
            .withColumn("is_deleted", lit("false"))
        )

        # align column order to target table
        target_cols = spark.table(target_table).columns
        df_final = df_final.select(*target_cols)

        # since you deleted all data and are starting fresh, overwrite is fine for full load
        # tables are truncated before full load, so append is safe
        df_final.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(target_table)

        rows_written = df_final.count()
        records_after = spark.table(target_table).count()

        # log ingested file FIRST
        log_ingested_file(
            run_id=run_id,
            table_name=table_name,
            file_path=file_path,
            rows_read=rows_read,
            rows_written=rows_written
        )

        # then finalize audit
        end_audit_success(
            run_id=run_id,
            rows_read=rows_read,
            rows_written=rows_written,
            rows_rejected=rows_rejected,
            records_before=records_before,
            records_after=records_after
        )

        print(f"{target_table}: {rows_written} rows loaded successfully")

    except Exception as e:
        end_audit_fail(run_id, str(e))
        raise

# =========================================================
# RUN
# =========================================================
for cfg in table_configs:
    load_to_bronze(cfg)